# Simplified Chinese support and reach on Steam

**DATA2010 Data Science Programming · final project**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theonlypgk/data2010-steam-chinese/blob/main/steam_chinese_analysis.ipynb)

**Main question:** should a Steam game support Simplified Chinese, for which genres, and when?
Reach = number of reviews. The notebook goes from the two raw files to the answers and
rebuilds every output table.

| Part | What it does | Output tables |
| :--- | :--- | :--- |
| [Setup](#scrollTo=setup) | connect the runtime to the project folder and check the package versions | |
| [Step 1 · Clean](#scrollTo=step1) | load the two snapshots, clean them, filter and encode | `pcd_games_clean` |
| [Step 2 · SQ1 groups](#scrollTo=step2) | group similar games, with vs without Chinese; test the Chinese-game filter | `pcd_sq1_groups` |
| [Step 3 · SQ2 groups](#scrollTo=step3) | group similar games, added Chinese after launch vs did not | `pcd_sq2_groups` |
| [Save and check](#scrollTo=save) | write the three tables, stop with an error if anything is wrong | |
| [Part B · Analysis](#scrollTo=partb) | figures and insights for EDA, SQ1, SQ2, SQ3 (reads the saved tables only) | `report/figures/` |

Tables are saved in `data/processed/` as csv (for code) and xlsx (to look at).

### How to run

- **Colab:** *Open in Colab* → *Runtime → Run all*. On a new runtime it stops once and asks
  for *Restart session*; then *Run all* runs to the end.
- **Own computer:** Python 3.12 or 3.13, `pip install -r requirements.txt`, *Run All*.

**Reproducibility**
- Run the cells in order, from Setup down.
- Only this notebook writes `data/processed/`; nothing is edited by hand.
- Inputs are checked by SHA-256 and package versions are pinned, so every run gives the same
  tables (the last cell prints their fingerprints).

**How to read it**
- Markdown at the top of a step: the strategy and why.
- Each code cell: one decision, with its label (1.2a, 1.2b, ...) and a one-line *Why*.
- `DATA ISSUE`: a problem found in the real data. **Check**: the numbers a correct run shows.

## Setup · Connect the runtime to the project

Run cells a-c first, on any machine.
- **a** finds the repository folder (on Colab: clones it) and works from there.
- **b** stops if the installed packages differ from `requirements.txt` (on Colab: installs them).
- **c** imports the libraries.

In [ ]:
# Setup a · Find the project folder
# Why: all later paths start from the project folder; on Colab the repository is cloned first.

import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/theonlypgk/data2010-steam-chinese"
IN_COLAB = "google.colab" in sys.modules           # True when the notebook runs on Google Colab
TESTED_PYTHON = {(3, 12), (3, 13)}                  # versions on which the pins were run end to end

if sys.version_info[:2] not in TESTED_PYTHON:
    raise RuntimeError(f"Python {sys.version.split()[0]}: use Python 3.12 or 3.13 (requirements.txt pins).")


def find_root():
    """First folder, from here upwards, that holds requirements.txt and this notebook."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "requirements.txt").exists() and (folder / "steam_chinese_analysis.ipynb").exists():
            return folder
    return None


ROOT = find_root()
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/data2010-steam-chinese")
    if not ROOT.exists():
        # GIT_LFS_SKIP_SMUDGE=1: leave the two large input files out of the clone;
        # Step 1.1 fetches them from the release, which has no Git LFS download quota.
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True,
                       env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
if ROOT is None:
    raise FileNotFoundError(f"Open the notebook inside the repository folder (git clone {REPO_URL}).")
os.chdir(ROOT)                                      # every relative path now starts here
print(f"Python {sys.version.split()[0]} | {'Google Colab' if IN_COLAB else 'local'} | project: {ROOT}")

In [ ]:
# Setup b · Use exactly the package versions of requirements.txt
# Why: another pandas or numpy version can give other numbers, so a mismatch stops the run.

NOT_NEEDED_TO_RUN = {"jupyter", "streamlit"}        # open the notebook / the app; not imported here


def pinned_versions() -> dict:
    """Package -> exact version from requirements.txt, for the packages this notebook imports."""
    pins = {}
    for line in (ROOT / "requirements.txt").read_text().splitlines():
        line = line.split("#")[0].strip()          # drop the comments
        if "==" in line:
            name, version = (part.strip() for part in line.split("=="))
            if name not in NOT_NEEDED_TO_RUN:
                pins[name] = version
    return pins


def installed_version(name: str):
    """Installed version of a package, or None if it is not installed."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


PINS = pinned_versions()
wrong = {name: pin for name, pin in PINS.items() if installed_version(name) != pin}
if wrong and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[f"{name}=={pin}" for name, pin in wrong.items()]], check=True)
    importlib.invalidate_caches()
    # A package that was already imported keeps its old version until the session restarts.
    loaded = [name for name in wrong if name.replace("-", "_") in sys.modules]
    if loaded:
        raise RuntimeError(f"Installed the pinned versions of {', '.join(loaded)}. "
                           "Choose Runtime > Restart session, then Runtime > Run all.")
    wrong = {name: pin for name, pin in PINS.items() if installed_version(name) != pin}
if wrong:
    found = {name: installed_version(name) for name in wrong}
    raise RuntimeError(f"Versions differ from requirements.txt: {found}. "
                       "Run pip install -r requirements.txt, restart the kernel, then Run All.")

print(" | ".join(f"{name} {pin}" for name, pin in PINS.items()))

In [ ]:
# Setup c · Import the libraries and name the project folders
# Why: all imports and folder names in one place.

import ast                    # literal_eval: read "['English', 'French']" as a Python list
import csv                    # read the raw header row on its own
import gzip                   # unpack the release copy of an input file
import hashlib                # SHA-256 fingerprints of the input and output files
import re                     # regular expressions for text cleaning and name matching
import shutil                 # copy files without loading them into memory
import urllib.request         # download the release copy of an input file
import zipfile                # rewrite the xlsx files with fixed dates
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

RAW = ROOT / "data" / "raw"                           # the two input files (Git LFS)
PROCESSED_CSV = ROOT / "data" / "processed" / "csv"   # output tables for code
PROCESSED_XLSX = ROOT / "data" / "processed" / "xlsx" # same tables, to open and look at
PREFIX = "pcd_"                                       # file prefix of every output table
print("imports done; raw inputs in", RAW)

# Step 1 · Clean the data and encode it → `pcd_games_clean`

Both snapshots are cleaned the same way; snapshot A becomes `pcd_games_clean`, snapshot C is
used again in Step 3.

| Sub-step | What |
|---|---|
| 1.1 Load | verify the two files (SHA-256), read 13 of 40 columns |
| 1.2 Clean | fix types and data issues; add outcome, treatment, confounders, flags; encode bins |
| 1.3 Filter | remove Chinese-first games, Early Access, < 10 reviews, < 1 year old, no genre/price/year |
| 1.4 Build | keep the columns of the cleaned dataset |
| 1.5 Check | funnel, removed games, random sample checked by hand |

## 1.1 Load the two raw snapshots

- **Snapshots:** A = 2026-06-20 (last version of `games.csv`), C = 2024-01-11, from Hugging Face
  `FronkonGames/steam-games-dataset`, pinned by commit.
- **Same bytes every run:** a file is used only if its SHA-256 matches. Tried in order:
  `data/raw/` (Git LFS) → GitHub release `raw-snapshots-v1` → Hugging Face.
- **DATA ISSUE:** the header of A merges two names (`DiscountDLC count`), which shifts every
  later column; it is repaired before reading.

In [ ]:
# 1.1a · Pin the input data and its fingerprints
# Why: fixed commits and SHA-256 fingerprints give the same input bytes on every run.

# Data source: two versions of the same file, pinned by commit hash.
HF_DATASET = "FronkonGames/steam-games-dataset"
HF_FILENAME = "games.csv"
REVISION = {
    "a": "52ba62abfa707a2fa31f4d147c2777a9d9fbd2da",   # snapshot A: last update of games.csv
    "c": "aefdeb1ff1a739b92ac5338682cc2610a0328505",   # snapshot C: January 2024 catalogue
}
SNAPSHOT_DATE_A = "2026-06-20"                          # day snapshot A was taken
SNAPSHOT_DATE_C = "2024-01-11"                          # day snapshot C was taken
# SHA-256 of each raw file; the same values are in SHA256SUMS.txt of the release.
SHA256 = {
    "a": "1b48008b01a799d82385d6e66ba0cb65dd477b6aa4b6c2ec8c443025ab6880ad",
    "c": "c755572b804a5c43f4f005aacef23cadfe92ae77d03daebf908d521c81285821",
}
RELEASE_URL = REPO_URL + "/releases/download/raw-snapshots-v1/games_{}.csv.gz"


def sha256_of(path) -> str:
    """SHA-256 fingerprint of a file, read in 1 MB blocks."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):   # 1 << 20 bytes = 1 MB; stops at b"" (end)
            digest.update(block)                          # add each block to the fingerprint
    return digest.hexdigest()

In [ ]:
# 1.1b · Fetch an input file that is missing or wrong
# Why: on Colab or without Git LFS the files are missing; release first (no quota), then Hugging Face.

def from_release(label: str, target) -> None:
    """Download the gzip copy of one snapshot from the GitHub release and unpack it."""
    packed = target.with_suffix(".csv.gz")                  # games_a.csv -> games_a.csv.gz
    urllib.request.urlretrieve(RELEASE_URL.format(label), packed)   # download the gzip copy
    with gzip.open(packed, "rb") as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)                       # unpack without loading it into memory
    packed.unlink()                                        # delete the .gz, keep the csv


def from_hugging_face(label: str, target) -> None:
    """Download games.csv of one snapshot from Hugging Face at its pinned commit."""
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(repo_id=HF_DATASET, filename=HF_FILENAME,
                             revision=REVISION[label], repo_type="dataset")
    shutil.copyfile(cached, target)                        # copy out of the download cache


# Order: data/raw copy, release, Hugging Face. Stops if none has the right fingerprint.
def ensure_snapshot(label: str):
    """Path of data/raw/games_<label>.csv, fetched if missing or wrong, always with the right SHA-256."""
    RAW.mkdir(parents=True, exist_ok=True)
    target = RAW / f"games_{label}.csv"
    if target.exists() and sha256_of(target) == SHA256[label]:
        print(f"snapshot {label.upper()}: data/raw copy verified")
        return target
    # try each source in order; the first copy with the right fingerprint is used
    for source, fetch in (("release raw-snapshots-v1", from_release),
                          ("Hugging Face", from_hugging_face)):
        try:
            fetch(label, target)
        except Exception as error:                          # no network, site down, ...
            print(f"snapshot {label.upper()}: {source} not reachable ({type(error).__name__})")
            continue
        if sha256_of(target) == SHA256[label]:
            print(f"snapshot {label.upper()}: downloaded from {source}, verified")
            return target
        print(f"snapshot {label.upper()}: copy from {source} has the wrong SHA-256")
    raise RuntimeError(f"snapshot {label.upper()}: no copy with SHA-256 {SHA256[label]} found")

In [ ]:
# 1.1c · Read only the 13 columns the analysis uses
# Why: the other 27 columns do not serve the questions.

# Raw column -> English name. The 13 columns read, and what each is for:
#   AppID                 id; joins snapshot C to A
#   Name                  to check results by eye
#   Release date          year band; ">= 1 year old" filter
#   Price, Discount       price tier (list price recovered from the sale price)
#   Supported languages   treatment, other-languages count, Chinese-first signal 1, languages added (Step 3)
#   Full audio languages  voice-over signal, only to test the filter (1.5b, 2d)
#   Positive, Negative    outcome: reach = positive + negative
#   Developers            Chinese-first signal 2
#   Publishers            shown in the 1.5 check only
#   Genres                Early Access flag
#   Tags                  genre group
# Not read (27): Estimated owners (ranges too wide); scores, playtime, CCU, achievements (quality
# or engagement, not reach); Categories, age, DLC, platforms, notes (not part of the design);
# descriptions, links and media (no numbers; text analysis is out of scope).
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Publishers": "publishers",
    "Genres": "genres",
    "Tags": "tags",
}
OPTIONAL_RAW_TO_CLEAN = {"Discount": "discount_pct"}   # only in snapshot A; "0" when missing

In [ ]:
# 1.1d · Repair the merged header of snapshot A (DATA ISSUE)
# Why: one header name covers two columns, which shifts every later column.

def read_header(raw_path) -> list[str]:
    """Header row of a raw file, with the merged name split back into two (DATA ISSUE)."""
    with open(raw_path, newline="", encoding="utf-8") as f:
        header = next(csv.reader(f))                       # first row only
    fixed = []
    for name in header:
        # "DiscountDLC count" -> ["Discount", "DLC count"]; any other name stays as it is
        fixed += ["Discount", "DLC count"] if name == "DiscountDLC count" else [name]
    return fixed

In [ ]:
# 1.1e · Read a snapshot as text, with English column names
# Why: everything is read as text; types are fixed in one place (1.2a).

def load_snapshot(raw_path) -> pd.DataFrame:
    """Read one raw snapshot: needed columns only, English names, every value as text."""
    names = read_header(raw_path)
    missing = [c for c in RAW_TO_CLEAN if c not in names]
    if missing:
        raise ValueError(f"{raw_path.name}: columns not found: {missing}")
    optional = [c for c in OPTIONAL_RAW_TO_CLEAN if c in names]
    raw = pd.read_csv(
        raw_path, names=names, header=None, skiprows=1,    # use the repaired header
        usecols=list(RAW_TO_CLEAN) + optional,             # read only these columns
        dtype=str, keep_default_na=False, na_values=[""],  # text; only "" counts as missing
        engine="c",
    )
    # Stop if columns are still shifted: language cells must look like "[...]" lists.
    langs = raw["Supported languages"].dropna()
    share_list_like = langs.str.startswith("[").mean() if len(langs) else 0
    if share_list_like < 0.9:
        raise ValueError(f"{raw_path.name}: columns look shifted; check read_header()")
    out = raw[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
    for raw_name, clean_name in OPTIONAL_RAW_TO_CLEAN.items():
        # copy the column when the snapshot has it, otherwise fill with "0"
        out[clean_name] = raw[raw_name] if raw_name in raw.columns else "0"
    return out

In [ ]:
# 1.1f · Run: fetch (if needed), verify and read both snapshots
# Why: both snapshots are loaded the same way.

raw_a = load_snapshot(ensure_snapshot("a"))
raw_c = load_snapshot(ensure_snapshot("c"))
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

**Check.** A has 125,855 games, C has 85,103; 13 columns each.

## 1.2 Clean each snapshot and add the analysis columns

No game is removed here (except rows without an id); 1.3 only uses the flags added here.

**Data issues fixed**
- **Sale prices:** A was taken during a sale (Hades 7.49 = 24.99 at -70%); `price` becomes the
  list price.
- **Junk in list cells:** store-page leftovers (`English[b]*[/b]`, HTML) are removed.
- **Ids and dates:** rows with no or a repeated id are dropped; unreadable dates become missing.

**Columns added**
- **Outcome:** `total_reviews = positive + negative`.
- **Treatment:** supports Simplified Chinese.
- **Confounders:** `genre_group` (first matching user tag), `price_tier`, `year_band`,
  `n_other_langs` (languages other than English and Chinese).

**Chinese-first games** (flagged here, removed in 1.3)
- *Why remove:* the question is about translating into Chinese; games whose original
  language is Chinese have Chinese players by default.
- *How:* no country column, so two signals computed from the data:
  1. Chinese text without English text (international games almost always have English).
  2. Every developer's name in Chinese characters, without kana (kana = Japanese).
- *Not used:* hand-made lists of cities or companies (incomplete, error-prone: "Paper Games"
  matches the British White Paper Games); voice-over languages (many Japanese games are
  dubbed in Chinese; right for 1 of 9 games in 1.5b).
- *Developers only, all of them:* publishers are often regional partners; a foreign game can
  list one Chinese co-developer.
- *Missed:* Chinese studios with an English name and English release (Black Myth: Wukong).
  Step 2d shows the result barely changes with or without the filter.

**Encoding.** Binned columns are saved as ranks, because text labels sort in the wrong order.
Ranks are not amounts, so they are never averaged; genre has no order and is not encoded.

| Column | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

In [ ]:
# 1.2a · Fix the types and the wrong single values (DATA ISSUES)
# Why: ids, counts, prices and dates arrive as text, some wrong (repeated ids, sale prices, two date formats).

# List price = sale price / (1 - discount / 100); discount capped at 99% (100% would divide by zero).
def fix_types(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Copy of one snapshot with real types, one row per id, list prices and dates."""
    df = trim.copy()
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")        # bad id -> NaN
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")        # DATA ISSUE: no / repeated id
    df["app_id"] = df["app_id"].astype("int64")
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # DATA ISSUE: sale prices in A -> list price = price / (1 - discount / 100)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # DATA ISSUE: dates come as "Aug 1, 2023" or "Aug 2023"; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days
    return df

In [ ]:
# 1.2b · Turn the list-like text cells into clean lists (DATA ISSUE)
# Why: languages, tags and genres are text that looks like a list, with store-page junk.

# DATA ISSUE: store-page leftovers in list cells: BBCode tags, HTML tags/entities, "*", line breaks
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


# Shapes handled: "['A', 'B']", "A,B", junk such as "English[b]*[/b]", missing. Order is kept (tags are sorted by votes).
def parse_list(value) -> list[str]:
    """List-like cell -> clean list of unique strings ("['A', 'B']", "A,B", junk, missing)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []                                          # missing cell
    text = _TAG_JUNK.sub("", str(value)).strip()           # remove the junk first
    if not text or text in {"[]", "nan"}:
        return []
    items = None
    if text.startswith("["):
        try:
            items = ast.literal_eval(text)                 # "['A', 'B']" -> ['A', 'B']
        except (ValueError, SyntaxError):
            text = text.strip("[]")                        # broken list: fall back to commas
    if items is None:
        items = text.split(",")                            # "A,B" -> ['A', 'B']
    cleaned = [str(i).strip().strip("'\"").strip() for i in items]
    seen, out = set(), []
    for item in cleaned:                                   # drop empties and duplicates,
        if item and item not in seen:                      # keep the original order
            seen.add(item)
            out.append(item)
    return out

In [ ]:
# 1.2c · Outcome, treatment and the other-languages confounder
# Why: reach = reviews (Steam publishes no sales); other languages also affect reach.

# Languages: the treatment, and the language every Western game ships with.
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"


def add_outcome_and_languages(df: pd.DataFrame, langs: pd.Series) -> pd.DataFrame:
    """Add total_reviews and the language columns."""
    df["total_reviews"] = df["positive"] + df["negative"]
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))   # set difference
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))                   # "English|French|..."
    return df

In [ ]:
# 1.2d · Flag the Chinese-first games
# Why: games whose original language is Chinese are not translations; no country column, so two signals from the data.

# Legal suffixes that become separate items when a developer cell is split on commas.
COMPANY_SUFFIXES = ["inc", "ltd", "llc", "co", "corp", "limited", "gmbh", "plc", "sa", "srl", "co. ltd"]


_HAN = re.compile(r"[一-鿿]")       # a Chinese character (also used in Japanese kanji)
_KANA = re.compile(r"[぀-ヿ]")      # hiragana or katakana: only Japanese uses them


def split_developers(text: str) -> list[str]:
    """"A, B, Inc." -> ["A", "B"]: split on commas and drop legal suffixes."""
    parts = [p.strip().strip(".").strip() for p in str(text).split(",")]   # split on commas, trim spaces and dots
    suffixes = set(COMPANY_SUFFIXES)
    return [p for p in parts if p and p.lower().rstrip(".") not in suffixes]   # drop empty parts and suffixes


# "All", not "any": a game made elsewhere can list one co-developer with a Chinese name.
def all_match(developers: list[str], pattern: re.Pattern) -> bool:
    """True if there is at least one developer and every developer matches `pattern`."""
    return bool(developers) and all(pattern.search(d) for d in developers)   # False for an empty list


# cn_text      Chinese text without English text                    (signal 1, used)
# cn_dev_name  all developer names in Chinese characters, no kana   (signal 2, used)
# cn_dev_kana  the same, with kana = Japanese                       (not used; tested in 1.5b, 2d)
# cn_audio     Chinese voice-over without English voice-over        (not used; tested in 1.5b, 2d)
def add_chinese_game_flags(df: pd.DataFrame, audio: pd.Series) -> pd.DataFrame:
    """Add the four signal columns, is_cn_game and the columns used to check it."""
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    developers = df["developers"].fillna("").astype(str)
    publishers = df["publishers"].fillna("").astype(str)
    dev_lists = developers.map(split_developers)
    # (Python's re via .map, not .str.contains: with pyarrow installed pandas uses Arrow's
    # regex engine, which rejects some of these patterns.)
    han_name = dev_lists.map(lambda d: all_match(d, _HAN))             # all developers: Chinese characters
    kana_name = developers.map(lambda text: bool(_KANA.search(text)))  # any kana in the developer cell
    has_zh_audio = audio.map(lambda x: bool(set(x) & chinese))
    has_en_audio = audio.map(lambda x: ENGLISH in x)

    df["cn_text"] = (df["has_zh_simplified"] | df["has_zh_traditional"]) & ~df["has_english"]
    df["cn_dev_name"] = han_name & ~kana_name
    df["cn_dev_kana"] = han_name & kana_name
    df["cn_audio"] = has_zh_audio & ~has_en_audio
    df["is_cn_game"] = df["cn_text"] | df["cn_dev_name"]               # the filter of 1.3

    used = df[["cn_text", "cn_dev_name"]]
    # names of the matched signals joined by ";", for the check in 1.5
    df["cn_rules"] = used.apply(lambda r: ";".join(r.index[r.to_numpy()]), axis=1)
    df["makers"] = developers + " | " + publishers
    return df

In [ ]:
# 1.2e · Flag Early Access and Traditional-Chinese-only games
# Why: unfinished games; and games that are neither "with Simplified Chinese" nor "without Chinese".

def add_filter_flags(df: pd.DataFrame, genres: pd.Series) -> pd.DataFrame:
    """Add is_early_access and is_traditional_only."""
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]
    return df

In [ ]:
# 1.2f · Give each game one genre group
# Why: genre changes audience size and amount of text; first matching tag wins (Steam orders tags by votes).

# Genre groups: user tag -> group; the first matching tag of a game wins.
TAG_TO_GENRE = {
    "Visual Novel": "Visual Novel",
    "RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Action RPG": "RPG", "Party-Based RPG": "RPG",
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "RTS": "Strategy",
    "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Action": "Action", "Shooter": "Action", "FPS": "Action", "Platformer": "Action",
    "Hack and Slash": "Action", "Fighting": "Action",
    "Puzzle": "Puzzle", "Hidden Object": "Puzzle",
    "Casual": "Casual",
}


def first_genre(tags: list[str]) -> str | None:
    """Group of the first tag that appears in TAG_TO_GENRE; None if no tag matches."""
    for tag in tags:
        if tag in TAG_TO_GENRE:
            return TAG_TO_GENRE[tag]
    return None

In [ ]:
# 1.2g · Bin price, release year and other languages, and encode the bins
# Why: groups need a few levels; ranks sort correctly, text labels do not ("10to20" before "5to10").

# Bins; each label is saved as its position 1, 2, 3, ... in the list.
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]    # USD, list price
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]           # codes 1-5
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]    # codes 1-4
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]                              # codes 1-3


def to_code(values: pd.Series, bins: list) -> pd.Series:
    """Bin numbers and return the bin's rank 1, 2, 3, ... (missing stays missing)."""
    # labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


def add_codes(df: pd.DataFrame) -> pd.DataFrame:
    """Add price_tier, year_band and n_other_langs_bin as ranks."""
    df["price_tier"] = to_code(df["price"], PRICE_BINS)                # codes 1-5
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)      # codes 1-4
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)   # codes 1-3
    return df

In [ ]:
# 1.2h · Run: clean both snapshots the same way
# Why: one function for both snapshots, so a game is measured the same way in A and C.

def add_features(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Cleaned copy of one snapshot with all derived columns."""
    df = fix_types(trim, snapshot_date)                               # 1.2a
    langs = df["supported_languages"].map(parse_list)                 # 1.2b
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)
    df = add_outcome_and_languages(df, langs)                         # 1.2c
    df = add_chinese_game_flags(df, audio)                            # 1.2d
    df = add_filter_flags(df, genres)                                 # 1.2e
    df["genre_group"] = tags.map(first_genre)                         # 1.2f
    return add_codes(df)                                              # 1.2g


feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

**Check.** No game is removed yet: A still has 125,855 games and C 85,103.

## 1.3 Filter the rows

Applied in this order; the funnel in 1.5 counts each step.

| Filter | Why |
|---|---|
| drop Chinese-first games | original language, not a translation |
| drop Early Access | unfinished; reviews behave differently |
| keep >= 10 reviews | below that, reach is noise (not higher: that would filter on the outcome) |
| keep released >= 1 year before the snapshot | time to collect reviews |
| keep games with genre, price and year | needed to form groups |
| `pcd_games_clean` only: drop Traditional-Chinese-only | neither "with" nor "without" Simplified Chinese |

In [ ]:
# 1.3a · Apply filters in order and count what each removes
# Why: named filters applied in order, so the funnel shows what each one removed.

def apply_filters(df: pd.DataFrame, steps: list[tuple[str, pd.Series]], table: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Keep rows passing every (name, mask) filter; return them and the funnel table."""
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)       # missing value = fails the filter
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)

In [ ]:
# 1.3b · The filters shared by both snapshots, and their thresholds
# Why: < 10 reviews is noise; < 1 year is too little time to collect reviews.

MIN_REVIEWS = 10              # below that, reach is noise (see the table above)
MIN_AGE_DAYS = 365            # every game had a year to collect reviews


def base_filters(df: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Filters used for both snapshots, as (name, True = keep) pairs, in order."""
    return [
        ("drop Chinese-first games (either of 2 signals)", ~df["is_cn_game"]),
        ("drop Early Access", ~df["is_early_access"]),
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

## 1.4 Build the cleaned dataset `pcd_games_clean`

One row per game of snapshot A that passed every filter. `has_zh`: 1 = Simplified Chinese,
0 = no Chinese. `log_total_reviews` is for figures (review counts are very skewed).

In [ ]:
# 1.4a · Build the cleaned dataset pcd_games_clean
# Why: one row per game that passed every filter, only the columns the analysis uses.

def build_games_clean(feat_a: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cleaned snapshot A: one row per game after all filters, and its funnel."""
    steps = base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
    df, funnel = apply_filters(feat_a, steps, "games_clean")
    df["has_zh"] = df["has_zh_simplified"].astype(int)               # True/False -> 1/0
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6) # rounded: same bytes every run
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        df[col] = df[col].astype(int)                                # no missing left after filters
    cols = [
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# 1.4b · Run: build the cleaned dataset
# Why: snapshot A is the current catalogue, used for SQ1.

games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

**Check.** `games_clean` has 45,445 games, 14,701 of them with Simplified Chinese.

## 1.5 Check what the cleaning removed

The funnel (games left after each filter) and the biggest removed Chinese-first games.
Shown only, not saved.

In [ ]:
# 1.5a · Show the funnel and the biggest Chinese-first games removed
# Why: no filter should remove most games; a wrongly removed game shows among the biggest.

cn_cols = ["app_id", "name", "makers", "cn_rules", "has_english", "total_reviews"]
cn_excluded = (
    feat_a[feat_a["is_cn_game"] & (feat_a["total_reviews"] >= MIN_REVIEWS)][cn_cols]  # would pass the review filter
    .sort_values("total_reviews", ascending=False)                                   # biggest first
    .reset_index(drop=True)
)
print(f"{len(cn_excluded):,} Chinese-first games removed (>= {MIN_REVIEWS} reviews)")
display(funnel_a)
cn_excluded.head(20)

**Why?** The biggest drop is the review filter (110,107 → 50,160): most Steam games have
fewer than 10 reviews. 2,030 Chinese-first games are removed; the top of the list is made in
China (Tale of Immortal, The Scroll Of Taiwu, The Hungry Lamb).

### 1.5b Hand check of a random sample

50 games drawn at random (fixed seed) from the games showing **any** of the four signals,
each checked by hand (developer name, store page). The last three columns are the hand check.

| Signal (first that applies) | In the filter? | Games | Chinese-first | Not | Unclear |
|---|---|---:|---:|---:|---:|
| `cn_text`: Chinese text, no English text | yes | 25 | 25 | 0 | 0 |
| `cn_dev_name`: developer names in Chinese characters, no kana | yes | 12 | 11 | 1 | 0 |
| `cn_dev_kana`: developer names with kana | no | 4 | 0 | 4 | 0 |
| `cn_audio`: Chinese voice-over, no English voice-over | no | 9 | 1 | 4 | 4 |

- **Filter signals:** right for 36 of 37 (the error: a Japanese studio with a kanji-only name).
- **`cn_text`:** 21 Chinese-language studios and 4 Chinese-only editions of foreign games; both
  are listings made for Chinese-speaking players.
- **Kana names and voice-over:** mostly wrong, so they are not used.
- A wrongly removed game only leaves the sample; it does not bias the comparison.

In [ ]:
# 1.5b · Draw the random sample that was checked by hand
# Why: a random sample shows how often each signal is right; fixed seed = same 50 games every run.

AUDIT_SIZE = 50               # games checked by hand
AUDIT_SEED = 2010             # fixed draw: the same sample on every run
SIGNALS = ["cn_text", "cn_dev_name", "cn_dev_kana", "cn_audio"]     # in the order of the table above
any_signal = feat_a[feat_a[SIGNALS].any(axis=1) & (feat_a["total_reviews"] >= MIN_REVIEWS)]
audit = any_signal.sample(n=AUDIT_SIZE, random_state=AUDIT_SEED).copy()
# idxmax on True/False columns gives the first column that is True: the first signal that applies
audit["signal"] = audit[SIGNALS].idxmax(axis=1)
print(f"{AUDIT_SIZE} of {len(any_signal):,} games with any signal, drawn with random_state={AUDIT_SEED}")
print(audit["signal"].value_counts().reindex(SIGNALS).to_string())
audit[["app_id", "name", "makers", "signal", "has_english", "total_reviews"]]

# Step 2 · SQ1 groups → `pcd_sq1_groups`

**Question.** How much larger is the reach of games with Simplified Chinese, and in which genres?

**Strategy.** Compare games with and without Chinese only inside groups of similar games.
Otherwise the gap would partly come from other differences (bigger games are translated more
and reviewed more).

**Group = genre x price tier x year band x other languages**

| Confounder | Why it is in the group |
|---|---|
| Genre | audience size and amount of text differ |
| Price tier | stands in for budget and studio size |
| Year band | older games had more time to collect reviews; bands, because single years leave only 47% of games in usable groups (79% with bands) |
| Other languages | the strongest: median 7 other languages with Chinese, 0 without (66% vs 9% have 5+) |

**Table `pcd_sq1_groups`** (one row per group)
- Games and **median** reviews on each side (medians: counts are very skewed).
- `pct_diff = median with Chinese / median without - 1`.
- `eligible`: >= 10 games on each side.
- Result: 360 groups, 190 eligible, 79% of games (87% of games with Chinese).

In [ ]:
# 2a · Define the SQ1 comparison group
# Why: games are compared only inside groups of similar games.

# code -> label, to write readable group names ("Action | <=2017 | lt5 | 5+")
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


def add_sq1_group(games: pd.DataFrame) -> pd.DataFrame:
    """Copy of games with group_id = genre | year band | price tier | other languages."""
    # assign returns a copy with the new column; the input table is not changed
    return games.assign(group_id=(
        games["genre_group"] + " | " + games["year_band"].map(YEAR_BAND_LABEL) + " | "
        + games["price_tier"].map(PRICE_LABEL) + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    ))

In [ ]:
# 2b · Compare games with and without Chinese inside each group
# Why: medians (skewed counts); >= 10 games per side, or one game moves the median.

MIN_GAMES_PER_SIDE = 10       # a group is eligible only with this many games per side (also Step 3)


# pct_diff = median with Chinese / median without - 1 (0.5 = +50% reviews).
def build_sq1_groups(games: pd.DataFrame) -> pd.DataFrame:
    """One row per group: games and median reviews with / without Chinese, gap, eligible."""
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")                    # count and median per group and side
        .unstack("has_zh")                                 # sides become columns: (stat, 0/1)
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)  # empty side -> 0
    out["pct_diff"] = (out["median_reviews_zh"] / out["median_reviews_no_zh"] - 1).round(6)
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)   # genre, then codes: old -> new, cheap -> expensive

In [ ]:
# 2c · Run: build the SQ1 group table
# Why: one row per group: the table SQ1 is answered from.

sq1_groups = build_sq1_groups(add_sq1_group(games_clean))
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

**Check.** 360 groups, 190 of them eligible.

## 2d Does the result depend on the Chinese-first filter?

The filter misses some games (Black Myth: Wukong). SQ1 is computed again with no filter, a
stricter filter, and without the biggest hits. Runs on copies; nothing is changed or saved.

In [ ]:
# 2d · Compute SQ1 again under other filters
# Why: if missed or wrongly removed games drove the result, it would move when the filter changes.

TOP_SHARE = 0.01              # share of most-reviewed games dropped in the last row


# Works on a copy (assign) and local variables: feat_a, games_clean and sq1_groups are not changed.
def sq1_summary(label: str, is_cn_game: pd.Series, drop_top: float = 0.0) -> dict:
    """SQ1 under the given Chinese-first flag, optionally without the most-reviewed games."""
    games, _ = build_games_clean(feat_a.assign(is_cn_game=is_cn_game))
    if drop_top:
        games = games[games["total_reviews"] <= games["total_reviews"].quantile(1 - drop_top)]
    groups = build_sq1_groups(add_sq1_group(games))
    eligible = groups[groups["eligible"]]
    return {"filter": label, "games": len(games), "eligible_groups": len(eligible),
            "median_pct_diff": round(float(eligible["pct_diff"].median()), 3)}


chosen = feat_a["cn_text"] | feat_a["cn_dev_name"]
stricter = chosen | feat_a["cn_dev_kana"] | feat_a["cn_audio"]
sensitivity = pd.DataFrame([
    sq1_summary("no filter", chosen & False),
    sq1_summary("chosen: text or developer name", chosen),
    sq1_summary("stricter: + kana names and voice-over", stricter),
    sq1_summary("chosen, without the 1% most-reviewed games", chosen, drop_top=TOP_SHARE),
])
sensitivity

**Why?** Median gap: +125% with no filter, +120% with the chosen filter, +117% with the
stricter one, +112% without the 1% most-reviewed games. The answer to SQ1 does not depend on
which Chinese-first games the filter catches or misses.

# Step 3 · SQ2 groups → `pcd_sq2_groups`

**Question.** Does adding Simplified Chinese after launch still increase reach, and in which genres?

**Strategy.** Follow the games that had no Chinese in January 2024 (C) to June 2026 (A).
Games that added Chinese (**treated**) are compared with similar games that added nothing
(**control**), because every game gains reviews over time anyway.

**3.1 The panel** (one row per game; not saved)
- Snapshot C, filtered as in Step 1; no Chinese in C; still present in A.
- Characteristics from C: what the game looked like before adding Chinese.
- Games with 0 reviews in A are dropped (delisted; -100% growth is not real).
- `growth = (reviews in A - reviews in C) / reviews in C`.
- `treated` = added Simplified Chinese. `control` = added neither Chinese nor a major language
  (German, French, Spanish, Japanese, Russian, Korean, Portuguese-Brazil). `excluded` = added
  another major language only (a localisation push of its own).

**3.2 Group = genre x year band.** Only 510 games added Chinese, so only the strongest confounders fit.

| Variable | In the group? | Why |
|---|---|---|
| Genre | yes | needed to answer per genre |
| Year band | yes | control games grow +12% (<=2017) vs +77% (2022+), and treated games are about 2 years newer |
| Price tier | no | little effect once the year is fixed; with it only 17 usable groups, 54% of treated games, 3 genres |
| Other languages | no | same mix in both arms (0 / 1-4 / 5+: 62/25/13% vs 64/26/10%) |

**Table `pcd_sq2_groups`**
- Treated and control games, their median growth, `uplift_pts = treated - control`
  (0.1 = +10 growth points); `eligible`: >= 10 games on each side.
- Result: 24 groups, 16 eligible, 90% of treated games, all 6 genres.

In [ ]:
# 3a · Treated, control or excluded: what a game added between C and A
# Why: a game that added other major languages had its own localisation push: not a clean control.

MAJOR_LANGS = [               # a control game must not have added any of these
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]


# Languages in A but not in C = added after January 2024.
def label_arm(a: pd.DataFrame, c: pd.DataFrame) -> np.ndarray:
    """treated / control / excluded for games without Chinese in C."""
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    added_zh = a["has_zh_simplified"].to_numpy()           # had no Chinese in C, has it in A
    added_major = np.array([bool(n & set(MAJOR_LANGS)) for n in new_langs])
    # np.select: first condition that is True gives the label, otherwise the default
    return np.select([added_zh, ~added_zh & ~added_major], ["treated", "control"], default="excluded")

In [ ]:
# 3b · Build the SQ2 panel: the same games followed from C to A
# Why: characteristics from C (before any change); delisted games dropped (-100% growth is not real).

def build_sq2_panel(feat_a: pd.DataFrame, feat_c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per game of C (no Chinese, filtered as in Step 1) also present in A; and its funnel."""
    steps = base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_panel")
    c = c.set_index("app_id")
    a = feat_a.set_index("app_id").loc[c.index]            # same games, same order, from A

    p = pd.DataFrame(index=c.index)
    for col in ["name", "genre_group", "release_year", "year_band", "price_tier"]:
        p[col] = c[col]                                    # characteristics BEFORE: from C
    p["price_c"] = c["price"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    p = p[p["reviews_a"] > 0].copy()                       # drop delisted games (0 reviews in A)
    a, c = a.loc[p.index], c.loc[p.index]
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)

    p["arm"] = label_arm(a, c)                             # 3a: treated / control / excluded
    for col in ["year_band", "price_tier"]:
        p[col] = p[col].astype(int)                        # no missing left after filters
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_panel", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_panel", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_panel", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    cols = ["name", "genre_group", "release_year", "year_band", "price_c", "price_tier",
            "reviews_c", "reviews_a", "growth", "arm"]
    return p[cols].reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# 3c · Run: build the panel and show its funnel
# Why: the funnel shows how many games each step keeps.

sq2_panel, funnel_c = build_sq2_panel(feat_a, feat_c)
print(f"sq2_panel: {len(sq2_panel):,} games")
print(sq2_panel["arm"].value_counts().to_string())
display(funnel_c)

**Check.** The panel has 21,590 games: 510 treated, 20,839 control and 241 excluded.

In [ ]:
# 3d · Compare treated and control growth inside each SQ2 group
# Why: group = genre x year band, the confounders that matter most for growth.

def add_sq2_group(panel: pd.DataFrame) -> pd.DataFrame:
    """Copy of the panel with group_id_sq2 = genre | year band."""
    # assign returns a copy with the new column; map turns the year code back into its label
    return panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))


# uplift_pts = median growth treated - median growth control.
def build_sq2_groups(panel: pd.DataFrame) -> pd.DataFrame:
    """One row per group: treated and control games, their median growth, uplift, eligible."""
    keys = ["group_id_sq2", "genre_group", "year_band"]
    used = panel[panel["arm"].isin(["treated", "control"])]          # "excluded" is not compared
    # count and median growth per group and arm; unstack makes treated / control columns
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)  # empty arm -> 0
    out["uplift_pts"] = (out["median_growth_treated"] - out["median_growth_control"]).round(6)
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)          # genre, then old -> new

In [ ]:
# 3e · Run: build the SQ2 group table
# Why: one row per group: the table SQ2 is answered from.

sq2_groups = build_sq2_groups(add_sq2_group(sq2_panel))
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

**Check.** 24 groups, 16 of them eligible.

# Save and check

- Tables are written as csv (for code) and xlsx (to look at, with fixed dates).
- The checks stop the notebook if anything is wrong, including the numbers in the **Check** notes.
- The last cell prints fingerprints of the saved tables: the same on every machine.

In [ ]:
# Save a · Write the three tables as csv and xlsx
# Why: csv for code, xlsx to look at; fixed dates keep the xlsx content the same on every run.

tables = {
    "games_clean": games_clean,   # Step 1: cleaned data, one row per game
    "sq1_groups": sq1_groups,     # Step 2: one row per SQ1 group
    "sq2_groups": sq2_groups,     # Step 3: one row per SQ2 group
}
FIXED_DATE = datetime(2026, 1, 1)                     # written into every xlsx instead of "now"


# An xlsx is a zip of XML files; the save time inside it and on each zip entry is replaced by FIXED_DATE.
def write_xlsx(table: pd.DataFrame, path, sheet: str) -> None:
    """Write an xlsx whose bytes depend only on the table: fixed dates inside and on the zip entries."""
    draft = path.with_name(path.stem + "_draft.xlsx")      # temporary file next to the final one
    with pd.ExcelWriter(draft, engine="openpyxl") as writer:
        table.to_excel(writer, index=False, sheet_name=sheet)
    stamp = FIXED_DATE.strftime("%Y-%m-%dT%H:%M:%SZ").encode()   # "2026-01-01T00:00:00Z" as bytes
    with zipfile.ZipFile(draft) as src, zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as dst:
        for item in src.infolist():
            data = src.read(item.filename)
            if item.filename == "docProps/core.xml":       # openpyxl writes the save time here
                data = re.sub(rb"(<dcterms:(?:created|modified)[^>]*>)[^<]*", rb"\g<1>" + stamp, data)
            entry = zipfile.ZipInfo(item.filename, date_time=FIXED_DATE.timetuple()[:6])
            entry.compress_type = zipfile.ZIP_DEFLATED
            dst.writestr(entry, data)                      # same entries, fixed timestamps
    draft.unlink()                                         # delete the draft


PROCESSED_CSV.mkdir(parents=True, exist_ok=True)
PROCESSED_XLSX.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV / f"{PREFIX}{name}.csv", index=False)
    write_xlsx(table, PROCESSED_XLSX / f"{PREFIX}{name}.xlsx", sheet=name[:31])   # sheet names max 31 chars
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

In [ ]:
# Save b · Check the structure of the tables and the saved files
# Why: catches broken ids, codes out of range, non-English names, xlsx different from csv.

# structure: unique ids, codes in range, English snake_case column names
assert games_clean["app_id"].is_unique and sq2_panel["app_id"].is_unique
assert set(games_clean["has_zh"]) <= {0, 1}
assert set(sq2_panel["arm"]) <= {"treated", "control", "excluded"}
assert games_clean["price_tier"].between(1, 5).all() and games_clean["year_band"].between(1, 4).all()
assert games_clean["n_other_langs_bin"].between(1, 3).all()
for name, table in tables.items():
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in table.columns), name

# saved files: the xlsx holds the same table as the csv
for name, table in tables.items():
    back = pd.read_excel(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx")
    assert back.shape == table.shape and list(back.columns) == list(table.columns), name

In [ ]:
# Save c · Check the data issues, the rules and the groups on known cases
# Why: known games show whether the fixes and rules behave as intended.

# known games: data issues and rules behave as intended
by_name = games_clean.set_index("name")
assert by_name.loc["Hades", "price"] > 20                             # list price, not the 7.49 sale price
assert by_name.loc["Hades", "price_tier"] == PRICE_LABELS.index("gt20") + 1
for name in ["Cyberpunk 2077", "Far Cry® 5", "Core Keeper", "Hollow Knight",   # Western games
             "Ether One", "STEINS;GATE"]:              # a name like a Chinese company; Japanese, dubbed in Chinese
    assert name in by_name.index, f"{name} should be kept"
for name in ["鬼谷八荒 Tale of Immortal", "The Scroll Of Taiwu", "Sword and Fairy 7"]:
    assert name in set(cn_excluded["name"]), f"{name} should be removed"  # signal 2, signal 1, both
assert sq2_panel["growth"].min() > -1                                 # no delisted game left

# group tables: every game in exactly one group, eligible groups really have >= 10 games per side
assert sq1_groups[["n_zh", "n_no_zh"]].to_numpy().sum() == len(games_clean)
assert sq2_groups[["n_treated", "n_control"]].to_numpy().sum() == (sq2_panel["arm"] != "excluded").sum()
assert (sq1_groups.loc[sq1_groups["eligible"], ["n_zh", "n_no_zh"]] >= MIN_GAMES_PER_SIDE).all().all()
assert (sq2_groups.loc[sq2_groups["eligible"], ["n_treated", "n_control"]] >= MIN_GAMES_PER_SIDE).all().all()

In [ ]:
# Save d · Check the numbers reported in the notes and the README
# Why: the text quotes these numbers; a rerun that gives others stops here.

# the run reproduces the numbers given in the Check notes and in the README
assert (len(raw_a), len(raw_c)) == (125_855, 85_103)
assert (len(games_clean), int(games_clean["has_zh"].sum())) == (45_445, 14_701)
assert len(cn_excluded) == 2_030
assert (len(sq1_groups), int(sq1_groups["eligible"].sum())) == (360, 190)
assert sq2_panel["arm"].value_counts().to_dict() == {"control": 20_839, "treated": 510, "excluded": 241}
assert (len(sq2_groups), int(sq2_groups["eligible"].sum())) == (24, 16)
# the sensitivity check: its "chosen" row is the main result, and the reported gaps
main_gap = round(float(sq1_groups.loc[sq1_groups["eligible"], "pct_diff"].median()), 3)
assert sensitivity.loc[1, "median_pct_diff"] == main_gap and sensitivity.loc[1, "games"] == len(games_clean)
assert sensitivity["median_pct_diff"].tolist() == [1.254, 1.2, 1.174, 1.121]
assert audit["signal"].value_counts().reindex(SIGNALS).tolist() == [25, 12, 4, 9]
print("all checks passed")

In [ ]:
# Save e · Print the fingerprints of the saved tables
# Why: two runs on any machine must print the same values.

# Fingerprints of the saved tables: two runs on any machine print the same values.
# content_fingerprint: fingerprint an output file by what it contains.
#   A csv is fingerprinted as it is. An xlsx is unzipped first and its entries are
#   fingerprinted in name order, because zip compression can differ between machines while
#   the content does not.
def content_fingerprint(path) -> str:
    """SHA-256 of a csv file, or of the unzipped entries of an xlsx file."""
    if path.suffix != ".xlsx":
        return sha256_of(path)
    digest = hashlib.sha256()
    with zipfile.ZipFile(path) as archive:                 # open the xlsx as the zip it is
        for name in sorted(archive.namelist()):
            digest.update(name.encode() + archive.read(name))   # entry name + its unzipped bytes
    return digest.hexdigest()


for path in sorted([*PROCESSED_CSV.glob(f"{PREFIX}*.csv"), *PROCESSED_XLSX.glob(f"{PREFIX}*.xlsx")]):
    print(f"{content_fingerprint(path)[:16]}  {path.relative_to(ROOT)}")

# Part B · Analysis: figures and insights

Reads only the three saved tables; figures go to `report/figures/`. To run it alone: Setup
a-c, then cell B a.

| Section | Reads |
| :--- | :--- |
| B1 EDA | `pcd_games_clean` |
| B2 SQ1 | `pcd_sq1_groups` |
| B3 SQ2 | `pcd_sq2_groups` |
| B4 SQ3 | `pcd_sq1_groups` and `pcd_sq2_groups`, per genre |

In [ ]:
# B a · Load the saved tables and prepare the figures folder
# Why: Part B reads only the saved tables, so it can run on its own.

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROCESSED_CSV = ROOT / "data" / "processed" / "csv"
PREFIX = "pcd_"
FIGURES = ROOT / "report" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


def load(name: str) -> pd.DataFrame:
    """Read one saved output table; stop with a clear message if Part A has not written it."""
    path = PROCESSED_CSV / f"{PREFIX}{name}.csv"           # e.g. data/processed/csv/pcd_games_clean.csv
    if not path.exists():
        raise FileNotFoundError(f"{path.relative_to(ROOT)} is missing: run Part A first.")
    return pd.read_csv(path)


games_clean = load("games_clean")
sq1_groups = load("sq1_groups")
sq2_groups = load("sq2_groups")
print(f"games_clean {len(games_clean):,} rows | sq1_groups {len(sq1_groups)} | sq2_groups {len(sq2_groups)}")

## B1. EDA

## B2. SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

## B3. SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

## B4. SQ3: for each genre, translate at launch, later, or not a priority?